# OPM Call/Price Range Optimizer

Reusable end-to-end notebook: given a spreadsheet with P/N, Total Calls, and DN Price columns, it:

1. Searches for **Call Range** breakpoints (A/B/C/D, given LA=2-3, C1=1, C0=0 are fixed) so that, across the whole dataset:
   - Count of P/N: `D < C < B < A < LA < C1 < C0` (increasing)
   - Total Calls: `D > C > B > A > LA > C1 > C0` (decreasing)
2. Searches for **DN Price** breakpoints (4 limits -> 5 bands) with the same increasing/decreasing pattern.
3. Runs an **alternating (joint) optimization**: Call Range and Price Range breakpoints are re-picked against each other a few times so the resulting Price x Call pivot grid has as few empty/lopsided cells as possible.
4. Among all the valid splits that satisfy the ordering above, prefers ones where **consecutive categories don't jump too far** in range width, Count of P/N, or Total Calls -- either as a soft preference, or as a hard cap you control with `MAX_JUMP_RATIO`.
5. Builds a new `.xlsx` workbook with the raw data, the breakpoints as editable blue inputs, and two live-formula pivot tables.


In [1]:
import numpy as np
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

FONT = "Arial"
CALL_CATS = ["D", "C", "B", "A", "LA", "C1", "C0"]

## Pivot scoring & smoothness helpers (used by the optimizer)

Two separate things get scored here:

- **Pivot quality** (`score_pivot`): how good the actual Price Range x Call Range grid looks for a given pair of breakpoints -- fewer empty cells, a bigger smallest cell, less imbalance overall.
- **Smoothness** (`max_step_ratio` / `call_smoothness` / `price_smoothness`): how big the *biggest single jump* is between one category and the next, checked three ways -- the width of the range, the Count of P/N in it, and the Total Calls in it. A smoothness score of `1.0` means every step is the same size; `2.0` means somewhere one category is twice the next; and so on.

The optimizer below uses pivot quality to judge candidates against each other, and uses smoothness both as a tie-breaker and, if you set `max_jump_ratio`, as a cap that filters out overly jumpy splits before that comparison happens.

In [2]:
def classify_calls(calls_arr, a_max, b_max, c_max):
    """Vectorized version of the Call Range logic used in the Excel formula."""
    cats = np.empty(len(calls_arr), dtype=object)
    calls_arr = np.asarray(calls_arr)
    cats[calls_arr == 0] = "C0"
    cats[calls_arr == 1] = "C1"
    cats[(calls_arr >= 2) & (calls_arr <= 3)] = "LA"
    cats[(calls_arr >= 4) & (calls_arr <= a_max)] = "A"
    cats[(calls_arr > a_max) & (calls_arr <= b_max)] = "B"
    cats[(calls_arr > b_max) & (calls_arr <= c_max)] = "C"
    cats[calls_arr > c_max] = "D"
    return cats


def classify_price(price_arr, p1, p2, p3, p4):
    """Vectorized version of the Price Range logic used in the Excel formula.
    Items with Price <= 0 (or missing) get their own "Empty" category instead
    of being dropped, so they still show up in the pivot / Grand Total."""
    price_arr = np.asarray(price_arr)
    cats = np.full(len(price_arr), "Empty", dtype=object)
    valid = price_arr > 0
    cats[valid & (price_arr <= p1)] = "to L1"
    cats[valid & (price_arr > p1) & (price_arr <= p2)] = "to L2"
    cats[valid & (price_arr > p2) & (price_arr <= p3)] = "to L3"
    cats[valid & (price_arr > p3) & (price_arr <= p4)] = "to L4"
    cats[valid & (price_arr > p4)] = "high value"
    return cats


PRICE_CATS = ["to L1", "to L2", "to L3", "to L4", "high value", "Empty"]


def build_pivot_counts(df, call_bps, price_bps):
    """Builds the Price Range x Call Range count grid for a given pair of
    breakpoints - the same grid PIVOT 1 in the output workbook shows.
    Includes an "Empty" price row so every P/N is represented."""
    a_max, b_max, c_max = call_bps
    p1, p2, p3, p4 = price_bps
    call_cats = classify_calls(df["Calls"].values, a_max, b_max, c_max)
    price_cats = classify_price(df["Price"].values, p1, p2, p3, p4)
    sub = pd.DataFrame({"CallCat": call_cats, "PriceCat": price_cats})
    pivot = pd.crosstab(sub["PriceCat"], sub["CallCat"])
    pivot = pivot.reindex(index=PRICE_CATS, columns=CALL_CATS, fill_value=0)
    return pivot


def score_pivot(pivot):
    """Lower is better. Primary: fewer empty cells. Secondary: a bigger smallest
    non-empty cell. Tertiary: less relative spread among the non-empty cells.
    The "Empty" price row is excluded from scoring since it isn't something
    either search can influence."""
    scoring_rows = [r for r in PRICE_CATS if r != "Empty"]
    flat = pivot.loc[scoring_rows].values.flatten()
    zero_cells = int((flat == 0).sum())
    nonzero = flat[flat > 0]
    if len(nonzero) == 0:
        return (zero_cells, 0, 0.0)
    min_nonzero = int(nonzero.min())
    mean_nonzero = float(nonzero.mean())
    cv = float(nonzero.std() / mean_nonzero) if mean_nonzero > 0 else 0.0
    return (zero_cells, -min_nonzero, cv)


def max_step_ratio(seq):
    """Largest ratio between any two consecutive elements in seq (>= 1).
    1.0 = perfectly smooth (every step the same size). Larger = a bigger
    jump somewhere in the sequence. Any zero/negative element makes a step
    undefined, treated as infinitely jumpy."""
    ratios = []
    for a, b in zip(seq[:-1], seq[1:]):
        if a <= 0 or b <= 0:
            return float("inf")
        ratios.append(max(a, b) / min(a, b))
    return max(ratios) if ratios else 1.0


def call_smoothness(a_max, b_max, c_max, cntA, cntB, cntC, cntD, sumA, sumB, sumC, sumD, LA_count, LA_sum):
    """Combined smoothness badness score (1.0 = smoothest) across the three
    things that matter: the width of each range, the Count of P/N in each
    category, and the Total Calls in each category. D is open-ended (no
    upper bound) so it's left out of the width check."""
    width_seq = [a_max - 3, b_max - a_max, c_max - b_max]          # A, B, C widths
    count_seq = [cntD, cntC, cntB, cntA, LA_count]                  # D -> LA
    sum_seq = [sumD, sumC, sumB, sumA, LA_sum]
    return max(max_step_ratio(width_seq), max_step_ratio(count_seq), max_step_ratio(sum_seq))


def price_smoothness(p1, p2, p3, p4, cnt1, cnt2, cnt3, cnt4, cnt5, sum1, sum2, sum3, sum4, sum5):
    """Same idea for the price side. "high value" is open-ended so it's left
    out of the width check; the first band's width is measured from 0."""
    width_seq = [p1, p2 - p1, p3 - p2, p4 - p3]
    count_seq = [cnt1, cnt2, cnt3, cnt4, cnt5]
    sum_seq = [sum1, sum2, sum3, sum4, sum5]
    return max(max_step_ratio(width_seq), max_step_ratio(count_seq), max_step_ratio(sum_seq))


def describe_smoothness(df, call_bps, price_bps):
    """Prints the width/count/Total-Calls sequence for both axes and the
    resulting smoothness score, so you can see exactly where -- if anywhere
    -- a big jump remains."""
    a_max, b_max, c_max = call_bps
    p1, p2, p3, p4 = price_bps

    calls_arr = df["Calls"].values
    def cband(lo, hi):
        mask = (calls_arr >= lo) & (calls_arr <= hi)
        return int(mask.sum()), int(calls_arr[mask].sum())
    LA_count, LA_sum = cband(2, 3)
    cntA, sumA = cband(4, a_max)
    cntB, sumB = cband(a_max + 1, b_max)
    cntC, sumC = cband(b_max + 1, c_max)
    cntD, sumD = cband(c_max + 1, int(calls_arr.max()))

    print("Call Range  (D, C, B, A, LA):")
    print(f"  widths (A,B,C only): {a_max-3}, {b_max-a_max}, {c_max-b_max}")
    print(f"  Count of P/N:        {cntD}, {cntC}, {cntB}, {cntA}, {LA_count}")
    print(f"  Total Calls:         {sumD}, {sumC}, {sumB}, {sumA}, {LA_sum}")
    print(f"  smoothness score:    {call_smoothness(a_max,b_max,c_max,cntA,cntB,cntC,cntD,sumA,sumB,sumC,sumD,LA_count,LA_sum):.2f}x")

    pdata = df[df["Price"] > 0].copy()
    def pband(lo, hi):
        mask = (pdata["Price"] > lo) & (pdata["Price"] <= hi)
        return int(mask.sum()), int(pdata.loc[mask, "Calls"].sum())
    c1, s1 = pband(0, p1); c2, s2 = pband(p1, p2); c3, s3 = pband(p2, p3)
    c4, s4 = pband(p3, p4); c5, s5 = pband(p4, float(pdata["Price"].max()))

    print("\nPrice Range (to L1..high value):")
    print(f"  widths (excl. high value): {p1}, {p2-p1}, {p3-p2}, {p4-p3}")
    print(f"  Count of P/N:               {c1}, {c2}, {c3}, {c4}, {c5}")
    print(f"  Total Calls:                {s1}, {s2}, {s3}, {s4}, {s5}")
    print(f"  smoothness score:           {price_smoothness(p1,p2,p3,p4,c1,c2,c3,c4,c5,s1,s2,s3,s4,s5):.2f}x")

## Data loading

In [3]:
def load_data(path, sheet=None, pn_col=None, price_col=None, calls_col=None):
    xls = pd.ExcelFile(path)
    sheet = sheet or xls.sheet_names[0]
    raw = pd.read_excel(path, sheet_name=sheet)

    def find_col(explicit, keyword):
        if explicit:
            return explicit
        for c in raw.columns:
            if keyword.lower() in str(c).lower():
                return c
        raise ValueError(f"Could not auto-detect a column matching '{keyword}'. "
                          f"Available columns: {list(raw.columns)}")

    pn_col = find_col(pn_col, "P/N")
    price_col = find_col(price_col, "Price")
    calls_col = find_col(calls_col, "Call")

    df = raw[[pn_col, price_col, calls_col]].copy()
    df.columns = ["PN", "Price", "Calls"]
    df["Price"] = pd.to_numeric(df["Price"], errors="coerce").fillna(0)
    df["Calls"] = pd.to_numeric(df["Calls"], errors="coerce").fillna(0).astype(int)
    return df

## Call Range breakpoint search (A/B/C/D boundaries, given LA/C1/C0 fixed)

`max_jump_ratio` (optional): a hard cap on how big a single step is allowed to be (in width, Count of P/N, or Total Calls) between consecutive categories. If no valid split meets the cap, the search restricts itself to the smoothest splits it actually found and ranks those by pivot quality, rather than silently ignoring the cap -- and tells you what the smoothest achievable value was.

In [4]:
def find_call_breakpoints(df, price_bps=None, max_jump_ratio=None):
    """
    Finds valid A/B/C/D call-range breakpoints.

    If `price_bps` is None, picks the smoothest breakpoint set using only
    the Calls distribution (the marginal search).

    If `price_bps` is given, evaluates candidate breakpoint sets (all of
    them still satisfy the base D<C<B<A<LA<C1<C0 count/calls pattern)
    against the resulting Price Range x Call Range pivot grid, and picks
    whichever pairs best with the given price_bps -- tie-broken by
    smoothness. This is the "call step" of the alternating joint
    optimization.

    `max_jump_ratio`: optional hard cap on the smoothness score (see
    call_smoothness). If given, candidates over the cap are dropped before
    ranking; if none of the candidates satisfy the cap, ranking falls back
    to a small pool of the smoothest splits actually available (never to
    the full unfiltered set), and prints what the best achievable value was.
    """
    LA_count = int(df["Calls"].isin([2, 3]).sum())
    LA_sum = int(df.loc[df["Calls"].isin([2, 3]), "Calls"].sum())

    sub = df[df["Calls"] >= 4]
    if sub.empty:
        raise ValueError("No items with Calls >= 4; cannot build A/B/C/D bands.")
    calls_arr = sub["Calls"].values
    vals = np.sort(sub["Calls"].unique()).tolist()

    def band_stats(lo, hi):
        mask = (calls_arr >= lo) & (calls_arr <= hi)
        return int(mask.sum()), int(calls_arr[mask].sum())

    results = []
    for ai, a_max in enumerate(vals):
        cntA, sumA = band_stats(4, a_max)
        if cntA >= LA_count or sumA <= LA_sum:
            continue
        for bi in range(ai + 1, len(vals)):
            b_max = vals[bi]
            cntB, sumB = band_stats(a_max + 1, b_max)
            if cntB >= cntA or sumB <= sumA:
                continue
            for ci in range(bi + 1, len(vals)):
                c_max = vals[ci]
                cntC, sumC = band_stats(b_max + 1, c_max)
                if cntC >= cntB or sumC <= sumB:
                    continue
                cntD, sumD = band_stats(c_max + 1, vals[-1])
                if cntD == 0 or cntD >= cntC or sumD <= sumC:
                    continue
                smoothness = call_smoothness(a_max, b_max, c_max, cntA, cntB, cntC, cntD,
                                              sumA, sumB, sumC, sumD, LA_count, LA_sum)
                results.append((smoothness, a_max, b_max, c_max))

    if not results:
        raise ValueError("No valid Call Range breakpoints found for this dataset "
                          "(the A/B/C/D/LA/C1/C0 monotonic pattern isn't achievable).")

    candidates = results
    if max_jump_ratio is not None:
        tight = [r for r in results if r[0] <= max_jump_ratio]
        if tight:
            candidates = tight
        else:
            results_sorted = sorted(results, key=lambda x: x[0])
            best_found = results_sorted[0][0]
            pool_size = max(10, len(results_sorted) // 20)
            candidates = results_sorted[:pool_size]
            print(f"  [Call Range] no split keeps every step's ratio <= {max_jump_ratio:.2f}x; "
                  f"restricting to the {pool_size} smoothest splits available "
                  f"(best = {best_found:.2f}x) before ranking by pivot quality.")

    if price_bps is None:
        candidates.sort(key=lambda x: x[0])
        _, a_max, b_max, c_max = candidates[0]
        return a_max, b_max, c_max

    scored = []
    for smoothness, a_max, b_max, c_max in candidates:
        pivot = build_pivot_counts(df, (a_max, b_max, c_max), price_bps)
        score = score_pivot(pivot)
        scored.append((score, smoothness, a_max, b_max, c_max))
    scored.sort(key=lambda x: (x[0], x[1]))
    _, _, a_max, b_max, c_max = scored[0]
    return a_max, b_max, c_max

## Price Range breakpoint search (4 limits -> 5 bands)

Same `max_jump_ratio` idea applies here. The search also now collects a handful of distinct valid splits per starting point (instead of stopping at the very first one it finds), so there are real alternatives to choose the smoothest -- or best-pairing -- one from.

In [5]:
def find_price_breakpoints(df, price_cap=None, whole_number=True, max_cap_search=200,
                            call_bps=None, max_jump_ratio=None):
    """
    Finds valid Price Range breakpoints (Limit of range 1-4).

    `price_cap` is optional - Limit of range 1 doesn't have to be any
    particular value. If given, it's used only as a soft starting point for
    the search; if None (the default), the search starts near the point
    that covers about 1/5 of the population and works outward from there.

    `whole_number`: if True (default), all 4 breakpoints come out as whole
    numbers.

    If `call_bps` is None, picks the smoothest solution using only the
    Price/Calls distribution (the marginal search).

    If `call_bps` is given, evaluates candidates against the resulting
    Price Range x Call Range pivot grid together with the given call_bps,
    and picks whichever pairs best -- tie-broken by smoothness. This is the
    "price step" of the alternating joint optimization.

    `max_jump_ratio`: optional hard cap on the smoothness score (see
    price_smoothness), same behaviour as in find_call_breakpoints -- falls
    back to a small pool of the smoothest splits actually available rather
    than the full unfiltered set if the cap can't be met.
    """
    pdata = df[df["Price"] > 0].copy()
    if pdata.empty:
        raise ValueError("No items with Price > 0; cannot build price bands.")

    if whole_number:
        pdata["Price"] = np.maximum(1, np.round(pdata["Price"].values)).astype(int)

    vals, inv = np.unique(pdata["Price"].values, return_inverse=True)
    cnt_per_val = np.bincount(inv, minlength=len(vals))
    sum_per_val = np.bincount(inv, weights=pdata["Calls"].values, minlength=len(vals))
    cum_cnt = np.cumsum(cnt_per_val)
    cum_sum = np.cumsum(sum_per_val)
    N = len(vals)
    total_cnt = int(cum_cnt[-1])
    total_sum = float(cum_sum[-1])

    def cum_at(i):
        if i < 0:
            return 0, 0.0
        return int(cum_cnt[i]), float(cum_sum[i])

    def band(prev_idx, idx):
        pc, ps = cum_at(prev_idx)
        c, s = cum_at(idx)
        return c - pc, s - ps

    def greedy_max(prev_idx, prev_cnt, prev_sum, min_start, max_idx):
        lo, hi, best = min_start, max_idx, None
        while lo <= hi:
            mid = (lo + hi) // 2
            bc, bs = band(prev_idx, mid)
            if bc < prev_cnt and bs < prev_sum and bc > 0:
                best = mid
                lo = mid + 1
            else:
                hi = mid - 1
        return best

    def dfs_from(idx1, max_solutions=6, tries_budget=150):
        """Collects up to `max_solutions` distinct valid (idx1,idx2,idx3,idx4)
        combos reachable from idx1 (instead of stopping at the very first
        one), so smoothness/pivot-quality can be judged across a few real
        alternatives rather than whatever the greedy search happens to find
        first."""
        solutions = []
        cnt1, sum1 = cum_at(idx1)
        if cnt1 == 0:
            return solutions
        max2 = greedy_max(idx1, cnt1, sum1, idx1 + 1, N - 2)
        if max2 is None:
            return solutions
        idx2, t2 = max2, 0
        while idx2 > idx1 and t2 < tries_budget and len(solutions) < max_solutions:
            cnt2, sum2 = band(idx1, idx2)
            if 0 < cnt2 < cnt1 and sum2 < sum1:
                max3 = greedy_max(idx2, cnt2, sum2, idx2 + 1, N - 2)
                if max3 is not None:
                    idx3, t3 = max3, 0
                    while idx3 > idx2 and t3 < tries_budget and len(solutions) < max_solutions:
                        cnt3, sum3 = band(idx2, idx3)
                        if 0 < cnt3 < cnt2 and sum3 < sum2:
                            max4 = greedy_max(idx3, cnt3, sum3, idx3 + 1, N - 1)
                            if max4 is not None:
                                idx4, t4 = max4, 0
                                while idx4 > idx3 and t4 < tries_budget and len(solutions) < max_solutions:
                                    cnt4, sum4 = band(idx3, idx4)
                                    if 0 < cnt4 < cnt3 and sum4 < sum3:
                                        cnt5 = total_cnt - cum_at(idx4)[0]
                                        sum5 = total_sum - cum_at(idx4)[1]
                                        if 0 < cnt5 < cnt4 and sum5 < sum4:
                                            solutions.append((idx1, idx2, idx3, idx4))
                                    idx4 -= 1
                                    t4 += 1
                        idx3 -= 1
                        t3 += 1
            idx2 -= 1
            t2 += 1
        return solutions

    def finalize(i1, i2, i3, i4):
        if whole_number:
            return int(vals[i1]), int(vals[i2]), int(vals[i3]), int(vals[i4])
        return float(vals[i1]), float(vals[i2]), float(vals[i3]), float(vals[i4])

    def counts_and_sums(i1, i2, i3, i4):
        c1, s1 = cum_at(i1)
        c2, s2 = cum_at(i2); cnt2, sum2 = c2 - c1, s2 - s1
        c3, s3 = cum_at(i3); cnt3, sum3 = c3 - c2, s3 - s2
        c4, s4 = cum_at(i4); cnt4, sum4 = c4 - c3, s4 - s3
        cnt5, sum5 = total_cnt - c4, total_sum - s4
        return (c1, cnt2, cnt3, cnt4, cnt5), (s1, sum2, sum3, sum4, sum5)

    target_idx1 = int(np.searchsorted(cum_cnt, max(1, total_cnt // 5), side="left"))
    if price_cap is not None:
        cap_idx_candidates = np.where(vals <= price_cap)[0]
        start_idx = cap_idx_candidates.max() if len(cap_idx_candidates) > 0 else target_idx1
    else:
        start_idx = target_idx1

    found = []
    for off in range(0, max_cap_search):
        idx1 = start_idx + off
        if idx1 >= N - 3:
            break
        found.extend(dfs_from(idx1))
        if call_bps is None and len(found) >= 40:
            break

    if not found:
        raise ValueError("No valid Price Range breakpoints found for this dataset.")

    scored = []
    for (i1, i2, i3, i4) in found:
        p1, p2, p3, p4 = finalize(i1, i2, i3, i4)
        (cnt1, cnt2, cnt3, cnt4, cnt5), (sum1, sum2, sum3, sum4, sum5) = counts_and_sums(i1, i2, i3, i4)
        smoothness = price_smoothness(p1, p2, p3, p4, cnt1, cnt2, cnt3, cnt4, cnt5,
                                       sum1, sum2, sum3, sum4, sum5)
        scored.append((smoothness, i1, p1, p2, p3, p4))

    candidates = scored
    if max_jump_ratio is not None:
        tight = [s for s in scored if s[0] <= max_jump_ratio]
        if tight:
            candidates = tight
        else:
            scored_sorted = sorted(scored, key=lambda x: x[0])
            best_found = scored_sorted[0][0]
            pool_size = max(10, len(scored_sorted) // 20)
            candidates = scored_sorted[:pool_size]
            print(f"  [Price Range] no split keeps every step's ratio <= {max_jump_ratio:.2f}x; "
                  f"restricting to the {pool_size} smoothest splits available "
                  f"(best = {best_found:.2f}x) before ranking by pivot quality.")

    if call_bps is None:
        candidates.sort(key=lambda x: (x[0], -x[1]))
        _, _, p1, p2, p3, p4 = candidates[0]
        return p1, p2, p3, p4

    rescored = []
    for smoothness, i1, p1, p2, p3, p4 in candidates:
        pivot = build_pivot_counts(df, call_bps, (p1, p2, p3, p4))
        pscore = score_pivot(pivot)
        rescored.append((pscore, smoothness, p1, p2, p3, p4))
    rescored.sort(key=lambda x: (x[0], x[1]))
    _, _, p1, p2, p3, p4 = rescored[0]
    return p1, p2, p3, p4

## Joint (alternating) optimization

Instead of picking Call Range and Price Range breakpoints independently, this alternates between the two searches, each time picking the option that pairs best with the other -- and, if `max_jump_ratio` is set, that also keeps every step-to-step jump (in range width, Count of P/N, and Total Calls) under that cap wherever a valid split allows it.

In [6]:
def optimize_joint(df, price_cap=None, whole_number=True, max_cap_search=200,
                    max_iters=10, max_jump_ratio=None, verbose=True):
    """
    Alternating (block coordinate ascent) optimization:
      1. Start from each axis's independent smoothest breakpoints.
      2. Re-pick Price breakpoints that pair best with the current Call
         breakpoints.
      3. Re-pick Call breakpoints that pair best with the new Price
         breakpoints.
      4. Repeat until neither changes, or max_iters is reached.

    `max_jump_ratio`: optional hard cap (see call_smoothness /
    price_smoothness) passed through to both searches at every step.

    Returns (call_bps, price_bps, n_iterations).
    """
    call_bps = find_call_breakpoints(df, max_jump_ratio=max_jump_ratio)
    price_bps = find_price_breakpoints(
        df, price_cap=price_cap, whole_number=whole_number,
        max_cap_search=max_cap_search, max_jump_ratio=max_jump_ratio
    )
    if verbose:
        print(f"Initial (independent) search -> Call={call_bps}, Price={price_bps}")

    for it in range(1, max_iters + 1):
        new_price_bps = find_price_breakpoints(
            df, price_cap=price_cap, whole_number=whole_number,
            max_cap_search=max_cap_search, call_bps=call_bps, max_jump_ratio=max_jump_ratio
        )
        new_call_bps = find_call_breakpoints(df, price_bps=new_price_bps, max_jump_ratio=max_jump_ratio)

        if verbose:
            print(f"Iteration {it} -> Call={new_call_bps}, Price={new_price_bps}")

        if new_call_bps == call_bps and new_price_bps == price_bps:
            call_bps, price_bps = new_call_bps, new_price_bps
            if verbose:
                print(f"Converged after {it} iteration(s).")
            return call_bps, price_bps, it

        call_bps, price_bps = new_call_bps, new_price_bps

    if verbose:
        print(f"Reached max_iters={max_iters} without full convergence; using the latest result.")
    return call_bps, price_bps, max_iters

## Workbook builder

In [ ]:
def build_workbook(df, call_bps, price_bps, output_path):
    a_max, b_max, c_max = call_bps
    p1, p2, p3, p4 = price_bps
    max_calls = int(df["Calls"].max())
    price_whole = all(isinstance(v, (int, np.integer)) for v in (p1, p2, p3, p4))
    price_number_format = "#,##0" if price_whole else "#,##0.00"

    # Sort by Tot Call descending so Accum./%Accum. are a meaningful running
    # total (matches the reference workbook's row order).
    df = df.sort_values("Calls", ascending=False, kind="mergesort").reset_index(drop=True)

    n = len(df)
    last_row = 1 + n

    wb = Workbook()
    ws = wb.active
    ws.title = "OPM_Optimized"

    blue = Font(name=FONT, color="0000FF")
    black = Font(name=FONT, color="000000")
    bold = Font(name=FONT, bold=True)
    header_fill = PatternFill("solid", start_color="D9E1F2")
    title_font = Font(name=FONT, bold=True, size=12)
    thin = Side(style="thin", color="BFBFBF")
    border = Border(left=thin, right=thin, top=thin, bottom=thin)

    # ------------------------------------------------------------------
    # Main data table
    # A=P/N, B=Tot Call, C=Accum., D=%Accum., E=RC, F=DN Price,
    # G=Call Range, H=Price Range
    # ------------------------------------------------------------------
    headers = ["P/N", "Tot Call", "Accum.", "%Accum.", "RC", "DN Price", "Call Range", "Price Range"]
    for i, h in enumerate(headers, start=1):
        c = ws.cell(row=1, column=i, value=h)
        c.font = bold
        c.fill = header_fill

    for idx, row in enumerate(df.itertuples(index=False), start=2):
        ws.cell(row=idx, column=1, value=row.PN).font = black

        ws.cell(row=idx, column=2, value=int(row.Calls)).font = black
        ws.cell(row=idx, column=2).number_format = "#,##0"

        accum_formula = f"=B{idx}" if idx == 2 else f"=C{idx - 1}+B{idx}"
        ws.cell(row=idx, column=3, value=accum_formula).font = black
        ws.cell(row=idx, column=3).number_format = "#,##0"

        ws.cell(row=idx, column=4, value=f"=C{idx}/$C${last_row}").font = black
        ws.cell(row=idx, column=4).number_format = "0.00%"

        ws.cell(row=idx, column=5,
                value=f'=IF(B{idx}=0,"D",IF(D{idx}<=50%,"A",IF(D{idx}<=80%,"B","C")))').font = black

        ws.cell(row=idx, column=6, value=float(row.Price)).font = black
        ws.cell(row=idx, column=6).number_format = "#,##0.00"

        ws.cell(row=idx, column=7,
                value=f'=IF(B{idx}=0,"C0",IF(B{idx}=1,"C1",IF(B{idx}<=3,"LA",'
                      f'IF(B{idx}<=$O$2,"A",IF(B{idx}<=$O$3,"B",IF(B{idx}<=$O$4,"C","D"))))))').font = black
        ws.cell(row=idx, column=8,
                value=f'=IF(OR(F{idx}="",F{idx}<=0),"Empty",IF(F{idx}<=$K$2,"to "&$K$2,'
                      f'IF(F{idx}<=$K$3,"to "&$K$3,IF(F{idx}<=$K$4,"to "&$K$4,'
                      f'IF(F{idx}<=$K$5,"to "&$K$5,"high value")))))').font = black

    for col, w in zip("ABCDEFGH", (16, 10, 12, 10, 6, 11, 12, 12)):
        ws.column_dimensions[col].width = w

    # ------------------------------------------------------------------
    # Breakpoints - Price Range first (aligns with Price Distribution below,
    # which starts at the same column), Call Range second (aligns with Call
    # Distribution below).
    # ------------------------------------------------------------------
    ws["J1"] = "DN Price Range"
    ws["J1"].font = title_font
    ws["J2"] = "Range 1"; ws["K2"] = p1
    ws["J3"] = "Range 2"; ws["K3"] = p2
    ws["J4"] = "Range 3"; ws["K4"] = p3
    ws["J5"] = "Range 4"; ws["K5"] = p4
    ws["J6"] = "Above Limit 4 = high value"
    ws["J7"] = "Items with DN Price <= 0 or blank = Empty"
    for r in (2, 3, 4, 5):
        ws.cell(row=r, column=11).font = blue
        ws.cell(row=r, column=11).number_format = price_number_format
    for r in (2, 3, 4, 5, 6, 7):
        ws.cell(row=r, column=10).font = Font(name=FONT, italic=(r in (6, 7)))
    ws.column_dimensions["J"].width = 22
    ws.column_dimensions["K"].width = 10

    ws["N1"] = "Call Range"
    ws["N1"].font = title_font
    ws["N2"] = "A"; ws["O2"] = a_max
    ws["N3"] = "B"; ws["O3"] = b_max
    ws["N4"] = "C"; ws["O4"] = c_max
    ws["N5"] = "D"; ws["O5"] = max_calls
    for r in (2, 3, 4, 5):
        ws.cell(row=r, column=15).font = blue
    for r in (2, 3, 4, 5, 6):
        ws.cell(row=r, column=14).font = Font(name=FONT, italic=(r == 6))
    ws.column_dimensions["N"].width = 34
    ws.column_dimensions["O"].width = 10

    data_rng_B = f"$B$2:$B${last_row}"
    data_rng_F = f"$F$2:$F${last_row}"
    data_rng_G = f"$G$2:$G${last_row}"
    data_rng_H = f"$H$2:$H${last_row}"

    # ------------------------------------------------------------------
    # Population Distribution (Price side + Call Range side, excludes C1/C0)
    # ------------------------------------------------------------------
    dist_title_row = 10
    dist_hdr_row = dist_title_row + 1
    dist_first_row = dist_hdr_row + 1

    ws.cell(row=dist_title_row, column=10, value="Population Distribution").font = title_font

    # -- Price Range distribution (columns J/K/L) --
    ws.cell(row=dist_hdr_row, column=10, value="Price Range").font = bold
    ws.cell(row=dist_hdr_row, column=11, value="Count").font = bold
    ws.cell(row=dist_hdr_row, column=12, value="% of Total").font = bold
    for j in (10, 11, 12):
        ws.cell(row=dist_hdr_row, column=j).fill = header_fill

    price_dist_labels = [f'="to "&$K$2', f'="to "&$K$3', f'="to "&$K$4', f'="to "&$K$5',
                          "high value", "Empty"]
    price_dist_formulas = [
        f'=COUNTIFS({data_rng_F},">0",{data_rng_F},"<="&$K$2,{data_rng_B},">1")',
        f'=COUNTIFS({data_rng_F},">"&$K$2,{data_rng_F},"<="&$K$3,{data_rng_B},">1")',
        f'=COUNTIFS({data_rng_F},">"&$K$3,{data_rng_F},"<="&$K$4,{data_rng_B},">1")',
        f'=COUNTIFS({data_rng_F},">"&$K$4,{data_rng_F},"<="&$K$5,{data_rng_B},">1")',
        f'=COUNTIFS({data_rng_F},">"&$K$5,{data_rng_B},">1")',
        f'=COUNTIFS({data_rng_F},"<=0",{data_rng_B},">1")',
    ]
    price_dist_last_row = dist_first_row + len(price_dist_labels) - 1
    price_dist_total_ref = f"$K${dist_first_row}:$K${price_dist_last_row}"
    for i, (lbl, formula) in enumerate(zip(price_dist_labels, price_dist_formulas)):
        r = dist_first_row + i
        ws.cell(row=r, column=10, value=lbl).font = black
        ws.cell(row=r, column=11, value=formula).font = black
        ws.cell(row=r, column=11).number_format = "#,##0"
        ws.cell(row=r, column=12, value=f"=K{r}/SUM({price_dist_total_ref})").font = black
        ws.cell(row=r, column=12).number_format = "0.00%"

    price_gt_row = price_dist_last_row + 1
    ws.cell(row=price_gt_row, column=10, value="Grand Total").font = bold
    ws.cell(row=price_gt_row, column=11, value=f"=SUM({price_dist_total_ref})").font = bold
    ws.cell(row=price_gt_row, column=11).number_format = "#,##0"
    ws.cell(row=price_gt_row, column=12, value=1).font = bold
    ws.cell(row=price_gt_row, column=12).number_format = "0.00%"

    # -- Call Range distribution (columns N/O/P), C1 and C0 excluded --
    ws.cell(row=dist_hdr_row, column=14, value="Call Range").font = bold
    ws.cell(row=dist_hdr_row, column=15, value="Count").font = bold
    ws.cell(row=dist_hdr_row, column=16, value="% of Total").font = bold
    for j in (14, 15, 16):
        ws.cell(row=dist_hdr_row, column=j).fill = header_fill

    call_dist_labels = ["D", "C", "B", "A", "LA"]
    call_dist_formulas = [
        f'=COUNTIFS({data_rng_B},">"&$O$4,{data_rng_B},"<="&$O$5,{data_rng_F},">0")',
        f'=COUNTIFS({data_rng_B},">"&$O$3,{data_rng_B},"<="&$O$4,{data_rng_F},">0")',
        f'=COUNTIFS({data_rng_B},">"&$O$2,{data_rng_B},"<="&$O$3,{data_rng_F},">0")',
        f'=COUNTIFS({data_rng_B},">3",{data_rng_B},"<="&$O$2,{data_rng_F},">0")',
        f'=COUNTIFS({data_rng_B},"<=3",{data_rng_B},">1",{data_rng_F},">0")',
    ]
    call_dist_last_row = dist_first_row + len(call_dist_labels) - 1
    call_dist_total_ref = f"$O${dist_first_row}:$O${call_dist_last_row}"
    for i, (lbl, formula) in enumerate(zip(call_dist_labels, call_dist_formulas)):
        r = dist_first_row + i
        ws.cell(row=r, column=14, value=lbl).font = black
        ws.cell(row=r, column=15, value=formula).font = black
        ws.cell(row=r, column=15).number_format = "#,##0"
        ws.cell(row=r, column=16, value=f"=O{r}/SUM({call_dist_total_ref})").font = black
        ws.cell(row=r, column=16).number_format = "0.00%"

    call_gt_row = price_gt_row  # align Grand Total rows visually
    ws.cell(row=call_gt_row, column=14, value="Grand Total").font = bold
    ws.cell(row=call_gt_row, column=15, value=f"=SUM({call_dist_total_ref})").font = bold
    ws.cell(row=call_gt_row, column=15).number_format = "#,##0"
    ws.cell(row=call_gt_row, column=16, value=1).font = bold
    ws.cell(row=call_gt_row, column=16).number_format = "0.00%"

    for r in range(dist_hdr_row, price_gt_row + 1):
        for j in (10, 11, 12):
            ws.cell(row=r, column=j).border = border
    for r in range(dist_hdr_row, call_gt_row + 1):
        for j in (14, 15, 16):
            ws.cell(row=r, column=j).border = border

    pivot_start_row = price_gt_row + 3

    # ------------------------------------------------------------------
    # Pivot tables (plain grids - population/% live in the block above)
    # ------------------------------------------------------------------
    def build_pivot(start_row, title, value_formula_maker):
        ws.cell(row=start_row, column=10, value=title).font = title_font
        hdr = start_row + 1
        ws.cell(row=hdr, column=10, value="Price Range \\ Call Range").font = bold
        ws.cell(row=hdr, column=10).fill = header_fill
        for j, cat in enumerate(CALL_CATS, start=11):
            c = ws.cell(row=hdr, column=j, value=cat)
            c.font = bold; c.fill = header_fill; c.alignment = Alignment(horizontal="center")
        gt_col = 11 + len(CALL_CATS)
        c = ws.cell(row=hdr, column=gt_col, value="Grand Total")
        c.font = bold; c.fill = header_fill

        price_labels = [f'="to "&$K$2', f'="to "&$K$3', f'="to "&$K$4', f'="to "&$K$5',
                         "high value", "Empty"]
        first_data_row = hdr + 1
        for i, lbl in enumerate(price_labels):
            r = first_data_row + i
            ws.cell(row=r, column=10, value=lbl).font = black
            for j, cat in enumerate(CALL_CATS, start=11):
                col_letter = get_column_letter(j)
                formula = value_formula_maker(col_letter, f"$J{r}", hdr)
                ws.cell(row=r, column=j, value=formula).font = black
                ws.cell(row=r, column=j).number_format = "#,##0"
            first_cat_col = get_column_letter(11)
            last_cat_col = get_column_letter(10 + len(CALL_CATS))
            ws.cell(row=r, column=gt_col, value=f"=SUM({first_cat_col}{r}:{last_cat_col}{r})").font = bold
            ws.cell(row=r, column=gt_col).number_format = "#,##0"

        gt_row = first_data_row + len(price_labels)
        last_data_row = first_data_row + len(price_labels) - 1

        ws.cell(row=gt_row, column=10, value="Grand Total").font = bold
        for j in range(11, gt_col + 1):
            col_letter = get_column_letter(j)
            ws.cell(row=gt_row, column=j,
                    value=f"=SUM({col_letter}{first_data_row}:{col_letter}{last_data_row})").font = bold
            ws.cell(row=gt_row, column=j).number_format = "#,##0"

        for r in range(hdr, gt_row + 1):
            for j in range(10, gt_col + 1):
                ws.cell(row=r, column=j).border = border
        return gt_row

    def make_count_formula(col_letter, price_ref, hdr_row):
        return f"=COUNTIFS({data_rng_G},{col_letter}${hdr_row},{data_rng_H},{price_ref})"

    def make_sum_formula(col_letter, price_ref, hdr_row):
        return f"=SUMIFS({data_rng_B},{data_rng_G},{col_letter}${hdr_row},{data_rng_H},{price_ref})"

    end1 = build_pivot(pivot_start_row, "PIVOT 1 : Count of P/N", make_count_formula)
    start2 = end1 + 3
    build_pivot(start2, "PIVOT 2 : Total Calls", make_sum_formula)

    for col in ["K", "L", "M", "N", "O", "P", "Q", "R"]:
        ws.column_dimensions[col].width = 12

    ws.freeze_panes = "A2"
    wb.save(output_path)
    print(f"Workbook written to {output_path}")

## Configuration

Edit the values below to match your input file, then run this cell and the **Execute** cell that follows.

- `INPUT_FILE`: path to your input .xlsx
- `OUTPUT_FILE`: path to write the result to
- `SHEET`, `PN_COL`, `PRICE_COL`, `CALLS_COL`: set these if auto-detection picks the wrong sheet/column
- `PRICE_CAP`: optional soft starting point for Limit of range 1 (e.g. `10`); leave as `None` to let the search pick freely
- `WHOLE_NUMBER_PRICE`: set `False` to allow decimal price breakpoints
- `MAX_ITERS`: how many alternating Call <-> Price rounds to run at most
- `MAX_JUMP_RATIO`: optional hard cap on how big a single step between consecutive categories is allowed to be, in range width, Count of P/N, *or* Total Calls (whichever jumps the most). `2.0` means "no category may be more than 2x its neighbour"; leave as `None` to only use smoothness as a soft tie-breaker instead of a hard rule. If the cap turns out to be infeasible for your data, the search tells you the best achievable value instead of silently ignoring it.

In [8]:
INPUT_FILE = "agc 19 call price 7Jul26.xlsx"
OUTPUT_FILE = "agc 19 output 7Jul26 1.1.xlsx"
SHEET = "Sheet3"        # or None for first sheet
PN_COL = "P/N"          # or None to auto-detect
PRICE_COL = "DN Price"  # or None to auto-detect
CALLS_COL = "TotCall"  # or None to auto-detect
PRICE_CAP = None            # optional soft starting point; no longer a hard requirement
WHOLE_NUMBER_PRICE = True    # set False to allow decimal price breakpoints
MAX_ITERS = 1000
MAX_JUMP_RATIO = None        # e.g. 2.0 to cap every step-to-step jump at 2x; None = soft preference only

## Execute

In [9]:
df = load_data(INPUT_FILE, SHEET, PN_COL, PRICE_COL, CALLS_COL)
print(f"Loaded {len(df)} rows.")

call_bps, price_bps, n_iters = optimize_joint(
    df, price_cap=PRICE_CAP, whole_number=WHOLE_NUMBER_PRICE,
    max_iters=MAX_ITERS, max_jump_ratio=MAX_JUMP_RATIO
)

print(f"\nFinal Call Range breakpoints -> A max={call_bps[0]}, B max={call_bps[1]}, C max={call_bps[2]}")
print(f"Final Price Range breakpoints -> {price_bps[0]}, {price_bps[1]}, {price_bps[2]}, {price_bps[3]}")

print()
describe_smoothness(df, call_bps, price_bps)

build_workbook(df, call_bps, price_bps, OUTPUT_FILE)

Loaded 25051 rows.
Initial (independent) search -> Call=(8, 19, 43), Price=(11, 43, 119, 483)
Iteration 1 -> Call=(6, 11, 22), Price=(11, 43, 119, 483)
Iteration 2 -> Call=(6, 11, 22), Price=(11, 43, 119, 483)
Converged after 2 iteration(s).

Final Call Range breakpoints -> A max=6, B max=11, C max=22
Final Price Range breakpoints -> 11, 43, 119, 483

Call Range  (D, C, B, A, LA):
  widths (A,B,C only): 3, 5, 11
  Count of P/N:        553, 593, 728, 994, 1548
  Total Calls:         32808, 9362, 6294, 4787, 3723
  smoothness score:    3.50x

Price Range (to L1..high value):
  widths (excl. high value): 11, 32, 76, 364
  Count of P/N:               5061, 5210, 5159, 5104, 4425
  Total Calls:                21026, 15742, 11052, 7375, 4107
  smoothness score:           4.79x
Workbook written to agc 19 output 7Jul26 1.1.xlsx
